# LDA で文書集合のトピックを見つける

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/10_topics_keywords/lda_topics.ipynb)

NLP クックブック 10-05 ／ トピックとキーワードの抽出

大量の文書がどんな話題で構成されているのかを、ラベルなしで把握したい。各文書がどの話題をどれくらい含むかも知りたい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import io, tarfile, urllib.request
import pandas as pd
from fugashi import Tagger

def load_livedoor():
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

df = load_livedoor().sample(3000, random_state=0).reset_index(drop=True)
tagger = Tagger()

def nouns(text):
    return " ".join(w.surface for w in tagger(text[:1500]) if w.feature.pos1 == "名詞" and w.feature.pos2 == "普通名詞" and len(w.surface) > 1)

df["nouns"] = df["text"].map(nouns)

In [3]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer(token_pattern=r"\S+", min_df=5, max_df=0.3)   # LDA には TF-IDF ではなく単語の出現回数を渡す
X = vec.fit_transform(df["nouns"])
lda = LatentDirichletAllocation(n_components=9, learning_method="batch", max_iter=30, random_state=0)
doc_topic = lda.fit_transform(X)        # 文書 × トピックの分布 (各行の和は 1)

vocab = vec.get_feature_names_out()
for k, weights in enumerate(lda.components_):
    print(k, " ".join(vocab[i] for i in weights.argsort()[::-1][:10]))

0 選手 代表 試合 監督 サッカー 野球 ゴルフ チーム スポーツ プロ
1 対応 搭載 フォン 発売 モデル 発表 機能 製品 カメラ 購入
2 話題 ビデオ iphone テレビ 発売 チェック 企業 売れ筋 登場 salon
3 自分 女性 結婚 男性 仕事 恋愛 相手 会社 ところ 独女
4 更新 ソフトウェア 利用 場合 sh 電話 phone アップデート 通信 ダウンロード
5 映画 公開 作品 世界 監督 本作 ドラマ 映像 登場 今回
6 ネット 掲示 番組 話題 情報 テレビ 放送 コメント ユーザー 批判
7 応募 女性 女子 プレゼント 当選 ファッション 人気 料理 クリスマス キャンペーン
8 アプリ 画面 表示 google android 機能 紹介 設定 情報 ゲーム


In [4]:
# 各トピックと元のカテゴリの対応を見る (答え合わせ)
df["topic"] = doc_topic.argmax(axis=1)
pd.crosstab(df["label"], df["topic"])

topic,0,1,2,3,4,5,6,7,8
label,,,,,,,,,
dokujo-tsushin,1,1,8,241,0,15,5,76,3
it-life-hack,5,139,36,0,11,10,26,7,109
kaden-channel,1,45,235,14,2,12,24,25,12
livedoor-homme,13,13,32,65,1,19,2,41,12
movie-enter,1,0,1,6,0,348,1,1,0
peachy,3,1,10,59,0,40,2,222,5
smax,0,143,3,0,77,0,0,2,130
sports-watch,276,0,0,9,0,4,77,1,0
topic-news,15,0,5,5,0,4,283,1,4


- トピックは「単語の確率分布」、文書は「トピックの確率分布」として表される。1 つの文書が複数のトピックを持てるのが、クラスタリングとの違い。
- トピック数は自分で決める。パープレキシティやコヒーレンスも参考になるが、最終的には「人間が見て解釈できるか」で選ぶ。
- 前処理が結果を大きく左右する。名詞に絞る、高頻度語・低頻度語を除く（`max_df`、`min_df`）のが基本。
- 大規模なコーパスや、コヒーレンスの計算・可視化 (pyLDAvis) まで行うなら gensim の `LdaModel` が定番。